# Perform significance testing on held-out test set

## Set Up

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../../"))
from src.config import BASE_PATH, SEED
from src.utils import BEST_PARAM_DICT, get_perm_seed
from src.sig import prep_train_test_model, get_baseline_reduced, get_p, format_p_val
import pandas as pd
from sklearn.metrics import r2_score
import numpy as np

In [ ]:
all_perm_dir = BASE_PATH / "results/perm_res/preds"
train_data_dir = BASE_PATH / "data"
test_data_dir = BASE_PATH / "data/test_data_toy"
model_dir = BASE_PATH / "models"
SHARED_DICT = {
    "max_iter": 50000,
    "seed": SEED,
    "linear_type": "ridge",
}
START_PERM = 0
END_PERM = 1000  # non-inclusive

## Hypotheses 1A/1B:

- Hypotheses:
	- H_A: r^2(<full_model>) - r^2(<reduced_model>) > 0
	- H_0: r^2(<full_model>) - r^2(<reduced_model>) <= 0

- A/B refers to self/parent cohorts

In [ ]:
null_dist_results_hyp1 = pd.DataFrame()
for residual_setup in ["SQT", "ST", "QT"]:
    for impact_type in ["self", "parent"]:
        X_train, y_train, X_test, y_test, model = prep_train_test_model(
            selected_params=BEST_PARAM_DICT[residual_setup][impact_type],
            impact_type=impact_type,
            train_data_dir=train_data_dir,
            test_data_dir=test_data_dir,
            shared_dict=SHARED_DICT,
        )

        ## GET BASELINE SCORE
        model.fit(X_train, y_train, shuffle=None, perm_dir=None)
        train_pred_all = model.predict_all(X_train)
        baseline_r2 = get_baseline_reduced(
            residual_setup, y_true=y_train, pred_df=train_pred_all
        )
        test_preds_no_shuffle = model.predict(X_test)
        r2_score_test_no_shuffle = r2_score(y_true=y_test, y_pred=test_preds_no_shuffle)
        r2_diff_no_shuffle = r2_score_test_no_shuffle - baseline_r2
        ## RUN THROUGH PERMS
        diff_list = []
        for perm_dir in all_perm_dir.iterdir():
            perm_name = perm_dir.name
            if perm_name == "baseline":
                continue
            # Set starting and end perm numbers to control where we are permuting
            perm_num = int(perm_name.split("_")[1].strip(""))
            if (perm_num < START_PERM) or (perm_num >= END_PERM):
                continue
            model.fit(X_train, y_train, shuffle="text", perm_dir=perm_dir)
            test_preds = model.predict(X_test)
            r2_score_test = r2_score(y_true=y_test, y_pred=test_preds)
            r2_diff_shuffled = r2_score_test - baseline_r2
            diff_list.append(r2_diff_shuffled)
            # TODO: Export perm iteration results to perm folder?
        null_dist_results_hyp1[f"{residual_setup}_{impact_type}"] = [
            r2_diff_no_shuffle
        ] + diff_list

null_dist_results_hyp1

In [ ]:
p_val_res_hyp1 = []
for impact_type in ["self", "parent"]:
    col_SQT = f"SQT_{impact_type}"
    col_ST = f"ST_{impact_type}"
    col_QT = f"QT_{impact_type}"
    df_sub = null_dist_results_hyp1[[col_SQT, col_ST, col_QT]]

    ## WESTFALL-YOUNG
    ## Iteration 1 --> max over ALL
    p_iter1, col_max1 = get_p(df_sub)
    ## Iteration 2 --> max over next 2
    df_sub = df_sub.drop(col_max1, axis=1)
    p_iter2, col_max2 = get_p(df_sub)
    ## Iteration 3 --> Just remaining sub-hypothesis
    df_sub = df_sub.drop(col_max2, axis=1)
    p_iter3, col_max3 = get_p(df_sub)

    p_val_res_hyp1.append(
        {
            "impact": impact_type,
            col_max1.split("_")[0]: format_p_val(p_iter1),
            col_max2.split("_")[0]: format_p_val(max([p_iter1, p_iter2])),
            col_max3.split("_")[0]: format_p_val(max([p_iter1, p_iter2, p_iter3])),
        }
    )

p_val_df_hyp1 = pd.DataFrame(p_val_res_hyp1)
p_val_df_hyp1

## Hypotheses 2A/2B:

- Hypotheses:
	- H_A: r^2(ST) - r^2(SQ) > 0
	- H_0: r^2(ST) - r^2(SQ) <= 0

- A/B refers to self/parent cohorts

In [ ]:
p_val_dict_hyp_2 = {}
null_dist_results_hyp_2 = pd.DataFrame()
for impact_type in ["self", "parent"]:
    ## Create and train model
    X_train_SQT, y_train_SQT, X_test_SQT, y_test_SQT, model_SQT = prep_train_test_model(
        selected_params=BEST_PARAM_DICT["SQT"][impact_type],
        impact_type=impact_type,
        train_data_dir=train_data_dir,
        test_data_dir=test_data_dir,
    )
    X_train_ST, y_train_ST, X_test_ST, y_test_ST, model_ST = prep_train_test_model(
        selected_params=BEST_PARAM_DICT["ST"][impact_type],
        impact_type=impact_type,
        train_data_dir=train_data_dir,
        test_data_dir=test_data_dir,
    )

    assert y_train_SQT.equals(y_train_ST)
    assert y_test_SQT.equals(y_test_ST)

    model_SQT.fit(X_train_SQT, y_train_SQT)
    model_ST.fit(X_train_ST, y_train_ST)

    #### 1) Get preds from each
    # 3-layer model- extract SQ layer preds from SQT model
    y_pred_SQ = model_SQT.predict_all(X_test_SQT)["pred_stage2_total"]
    # 2-layer model - ST is final model output here, no need to extract
    y_pred_ST = model_ST.predict(X_test_ST)

    base_r2_SQ = r2_score(y_true=y_test_SQT, y_pred=y_pred_SQ)
    base_r2_ST = r2_score(y_true=y_test_ST, y_pred=y_pred_ST)

    baseline_diff = base_r2_ST - base_r2_SQ
    ### 2) FOR EACH perm-- randomly swap preds
    diff_list = []
    for perm_dir in all_perm_dir.iterdir():
        perm_name = perm_dir.name
        if perm_name == "baseline":
            continue
        # Set starting and end perm numbers to control where we are permuting
        perm_num = int(perm_name.split("_")[1].strip(""))
        if (perm_num < START_PERM) or (perm_num >= END_PERM):
            continue
        perm_seed = get_perm_seed(perm_dir=perm_dir)
        rng = np.random.RandomState(perm_num)
        mask = rng.rand(len(y_test_SQT)) < 0.5
        perm_SQ = np.where(mask, y_pred_SQ, y_pred_ST)
        perm_ST = np.where(mask, y_pred_ST, y_pred_SQ)

        diff_list.append(
            r2_score(y_true=y_test_SQT, y_pred=perm_ST)
            - r2_score(y_true=y_test_SQT, y_pred=perm_SQ)
        )
        # TODO: Export perm iteration results to perm folder?
    null_dist_results_hyp_2[impact_type] = [baseline_diff] + diff_list
    ### 3) Get p-val
    p_val = (np.array(diff_list) >= baseline_diff).sum() / (len(diff_list) + 1)
    p_val_dict_hyp_2[impact_type] = format_p_val(p_val)